 Part 4: Statistical Arbitrage Overlay

---

## 📋 Project Overview

**Objective:** Identify co-moving assets and build a statistical arbitrage overlay for the portfolio.



---

### Part 4 Goals:
1. **Correlation Analysis** - Find highly correlated asset pairs
2. **Cointegration Testing** - Identify statistically cointegrated pairs
3. **Lead-Lag Relationships** - Discover which assets lead/lag others
4. **Pairs Trading Strategy** - Design a mean-reversion trading approach
5. **Integration Proposal** - How to incorporate into main portfolio

---

### What is Statistical Arbitrage?

> Statistical arbitrage exploits temporary pricing inefficiencies between related assets. When two assets that normally move together diverge, we bet on their convergence.

**Key Concepts:**
- **Correlation:** Assets that move together (but may diverge temporarily)
- **Cointegration:** A stronger relationship - the spread between assets is mean-reverting
- **Lead-Lag:** One asset's movement predicts another's future movement

---

## 📦 Step 1: Import Libraries

In [ ]:
# Install required packages (uncomment for Colab)
# !pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn --quiet

# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Statistical Analysis
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

# Econometrics
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.api import VAR
import statsmodels.api as sm

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📥 Step 2: Load Data

In [ ]:
# Load price data
try:
    prices = pd.read_csv('daily_prices.csv')
    print(f"✅ Price data loaded: {len(prices):,} rows")
except FileNotFoundError:
    try:
        prices = pd.read_csv('processed_features.csv')
        print(f"✅ Processed data loaded: {len(prices):,} rows")
    except FileNotFoundError:
        print("❌ Data file not found. Please upload the data.")

In [ ]:
# Identify columns
ticker_col = date_col = close_col = None

for col in prices.columns:
    col_lower = col.lower()
    if col_lower in ['ticker', 'symbol', 'stock', 'asset']:
        ticker_col = col
    elif col_lower in ['date', 'datetime', 'time']:
        date_col = col
    elif col_lower in ['close', 'adj_close', 'adjusted_close']:
        close_col = col

print(f"📌 Columns: Ticker={ticker_col}, Date={date_col}, Close={close_col}")

# Convert date
prices[date_col] = pd.to_datetime(prices[date_col])

# Create price matrix (pivot)
price_matrix = prices.pivot(index=date_col, columns=ticker_col, values=close_col)
price_matrix = price_matrix.sort_index()

# Calculate returns
returns_matrix = price_matrix.pct_change().dropna()

print(f"\n📊 Price matrix shape: {price_matrix.shape}")
print(f"📊 Returns matrix shape: {returns_matrix.shape}")
print(f"📅 Date range: {price_matrix.index[0]} to {price_matrix.index[-1]}")
print(f"🏢 Number of assets: {price_matrix.shape[1]}")

---

## 📊 Step 3: Correlation Analysis

We'll start with basic correlation analysis to identify assets that move together.

In [ ]:
# Calculate correlation matrix
print("📊 Calculating Correlation Matrix...")

# Use returns for correlation (more stationary than prices)
corr_matrix = returns_matrix.corr()

print(f"✅ Correlation matrix shape: {corr_matrix.shape}")

# Basic statistics
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
correlations = upper_triangle.stack()

print(f"\n📈 Correlation Statistics:")
print(f"   Mean: {correlations.mean():.4f}")
print(f"   Std: {correlations.std():.4f}")
print(f"   Min: {correlations.min():.4f}")
print(f"   Max: {correlations.max():.4f}")
print(f"   Median: {correlations.median():.4f}")

In [ ]:
# Visualize correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, ax=axes[0], 
            xticklabels=True, yticklabels=True, square=True,
            cbar_kws={'label': 'Correlation'})
axes[0].set_title('Correlation Matrix Heatmap', fontweight='bold', fontsize=12)

# Distribution of correlations
axes[1].hist(correlations, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=correlations.mean(), color='green', linestyle='-', linewidth=2, 
                 label=f'Mean: {correlations.mean():.3f}')
axes[1].set_title('Distribution of Pairwise Correlations', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Correlation')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Find most correlated pairs
print("📊 Most Correlated Asset Pairs:")
print("="*60)

# Get all pairs with their correlations
pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        pairs.append({
            'Asset1': corr_matrix.columns[i],
            'Asset2': corr_matrix.columns[j],
            'Correlation': corr_matrix.iloc[i, j]
        })

pairs_df = pd.DataFrame(pairs)
pairs_df = pairs_df.sort_values('Correlation', ascending=False)

print("\n🏆 Top 15 Most Positively Correlated Pairs:")
display(pairs_df.head(15))

print("\n🔻 Top 15 Most Negatively Correlated Pairs:")
display(pairs_df.tail(15))

---

## 📈 Step 4: Cointegration Analysis

Cointegration is a stronger relationship than correlation. Two assets are cointegrated if their spread is stationary (mean-reverting), even if the individual price series are non-stationary.

**Why cointegration for pairs trading?**
- A mean-reverting spread can be traded: short when high, long when low
- More robust than correlation (correlation can break down)
- Has a clear statistical framework

In [ ]:
def test_cointegration(price1, price2, significance=0.05):
    """
    Test cointegration between two price series.
    
    Returns:
    - t_stat: test statistic
    - p_value: p-value of the test
    - coint_flag: True if cointegrated at significance level
    - hedge_ratio: optimal hedge ratio (beta)
    """
    # Remove NaN
    valid_idx = ~(price1.isna() | price2.isna())
    p1 = price1[valid_idx].values
    p2 = price2[valid_idx].values
    
    if len(p1) < 100:  # Need sufficient data
        return np.nan, np.nan, False, np.nan
    
    try:
        # Engle-Granger cointegration test
        t_stat, p_value, _ = coint(p1, p2)
        
        # Calculate hedge ratio using OLS
        X = sm.add_constant(p2)
        model = OLS(p1, X).fit()
        hedge_ratio = model.params[1]
        
        coint_flag = p_value < significance
        
        return t_stat, p_value, coint_flag, hedge_ratio
    except:
        return np.nan, np.nan, False, np.nan

print("✅ Cointegration test function defined!")

In [ ]:
# Test cointegration for all pairs
print("📊 Testing Cointegration for All Pairs...")
print("(This may take a few minutes)")

tickers = price_matrix.columns.tolist()
n_tickers = len(tickers)

coint_results = []

total_pairs = n_tickers * (n_tickers - 1) // 2
checked = 0

for i in range(n_tickers):
    for j in range(i+1, n_tickers):
        t1, t2 = tickers[i], tickers[j]
        
        t_stat, p_value, is_coint, hedge_ratio = test_cointegration(
            price_matrix[t1], price_matrix[t2]
        )
        
        # Get correlation
        corr = returns_matrix[t1].corr(returns_matrix[t2])
        
        coint_results.append({
            'Asset1': t1,
            'Asset2': t2,
            't_stat': t_stat,
            'p_value': p_value,
            'is_cointegrated': is_coint,
            'hedge_ratio': hedge_ratio,
            'correlation': corr
        })
        
        checked += 1
        if checked % 100 == 0:
            print(f"   Checked {checked}/{total_pairs} pairs...")

coint_df = pd.DataFrame(coint_results)
print(f"\n✅ Cointegration testing complete!")
print(f"   Total pairs tested: {len(coint_df)}")
print(f"   Cointegrated pairs (p<0.05): {coint_df['is_cointegrated'].sum()}")

In [ ]:
# Display cointegrated pairs
print("📊 Cointegrated Asset Pairs (p < 0.05):")
print("="*60)

coint_pairs = coint_df[coint_df['is_cointegrated']].sort_values('p_value')

if len(coint_pairs) > 0:
    display(coint_pairs.head(20))
else:
    print("No cointegrated pairs found at 5% significance level.")
    print("Showing pairs with lowest p-values:")
    display(coint_df.sort_values('p_value').head(20))

In [ ]:
# Visualize top cointegrated pairs
print("📈 Visualizing Top Cointegrated/Correlated Pairs:")

# Select top 4 pairs to visualize
if len(coint_pairs) >= 4:
    top_pairs = coint_pairs.head(4)
else:
    top_pairs = coint_df.sort_values('p_value').head(4)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (_, row) in enumerate(top_pairs.iterrows()):
    if idx >= 4:
        break
    
    asset1, asset2 = row['Asset1'], row['Asset2']
    
    # Normalize prices to 100
    p1_norm = price_matrix[asset1] / price_matrix[asset1].iloc[0] * 100
    p2_norm = price_matrix[asset2] / price_matrix[asset2].iloc[0] * 100
    
    axes[idx].plot(p1_norm.index, p1_norm.values, label=asset1, linewidth=1.5)
    axes[idx].plot(p2_norm.index, p2_norm.values, label=asset2, linewidth=1.5)
    axes[idx].set_title(f'{asset1} vs {asset2}\nCorr: {row["correlation"]:.3f}, p-value: {row["p_value"]:.4f}',
                        fontweight='bold')
    axes[idx].set_xlabel('Date')
    axes[idx].set_ylabel('Normalized Price (Base=100)')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Top Cointegrated/Correlated Pairs (Normalized Prices)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 🔄 Step 5: Lead-Lag Relationship Analysis

Some assets may lead others - their price movements today predict another asset's movement tomorrow.

In [ ]:
def find_lead_lag(returns1, returns2, max_lag=10):
    """
    Find lead-lag relationship between two return series.
    
    Returns:
    - best_lag: lag with highest absolute correlation
    - correlations: dict of lag -> correlation
    """
    correlations = {}
    
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            # Asset1 leads Asset2 by |lag| days
            r1 = returns1.iloc[:lag].values
            r2 = returns2.iloc[-lag:].values
        elif lag > 0:
            # Asset2 leads Asset1 by lag days
            r1 = returns1.iloc[lag:].values
            r2 = returns2.iloc[:-lag].values
        else:
            r1 = returns1.values
            r2 = returns2.values
        
        if len(r1) > 10:
            corr, _ = pearsonr(r1, r2)
            correlations[lag] = corr
    
    best_lag = max(correlations, key=lambda x: abs(correlations[x]))
    
    return best_lag, correlations

print("✅ Lead-lag analysis function defined!")

In [ ]:
# Analyze lead-lag for top pairs
print("📊 Lead-Lag Relationship Analysis:")
print("="*60)

# Use top correlated or cointegrated pairs
if len(coint_pairs) > 0:
    analysis_pairs = coint_pairs.head(10)
else:
    analysis_pairs = pairs_df.head(10)

lead_lag_results = []

for _, row in analysis_pairs.iterrows():
    asset1, asset2 = row['Asset1'], row['Asset2']
    
    best_lag, corrs = find_lead_lag(
        returns_matrix[asset1], 
        returns_matrix[asset2],
        max_lag=5
    )
    
    lead_lag_results.append({
        'Asset1': asset1,
        'Asset2': asset2,
        'Best_Lag': best_lag,
        'Lag_Correlation': corrs.get(best_lag, np.nan),
        'Contemporaneous_Correlation': corrs.get(0, np.nan),
        'Leader': asset1 if best_lag > 0 else (asset2 if best_lag < 0 else 'Neither')
    })

lead_lag_df = pd.DataFrame(lead_lag_results)
print("\n📈 Lead-Lag Results:")
display(lead_lag_df)

In [ ]:
# Visualize lead-lag for one pair
if len(analysis_pairs) > 0:
    sample_pair = analysis_pairs.iloc[0]
    asset1, asset2 = sample_pair['Asset1'], sample_pair['Asset2']
    
    best_lag, corrs = find_lead_lag(
        returns_matrix[asset1], 
        returns_matrix[asset2],
        max_lag=10
    )
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    lags = sorted(corrs.keys())
    corr_vals = [corrs[l] for l in lags]
    
    colors = ['green' if l == best_lag else 'steelblue' for l in lags]
    
    ax.bar(lags, corr_vals, color=colors, edgecolor='black', alpha=0.7)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel(f'Lag (positive = {asset2} leads {asset1})', fontsize=11)
    ax.set_ylabel('Correlation', fontsize=11)
    ax.set_title(f'Lead-Lag Correlation: {asset1} vs {asset2}\nBest lag: {best_lag}', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

---

## 📊 Step 6: Spread Analysis for Pairs Trading

For cointegrated pairs, we can construct a spread that should be mean-reverting.

In [ ]:
def calculate_spread(price1, price2, hedge_ratio):
    """
    Calculate the spread between two assets.
    Spread = Price1 - hedge_ratio * Price2
    """
    return price1 - hedge_ratio * price2

def calculate_zscore(spread, lookback=20):
    """
    Calculate rolling z-score of the spread.
    """
    rolling_mean = spread.rolling(lookback).mean()
    rolling_std = spread.rolling(lookback).std()
    return (spread - rolling_mean) / rolling_std

print("✅ Spread functions defined!")

In [ ]:
# Analyze spread for the best pair
if len(coint_pairs) > 0:
    best_pair = coint_pairs.iloc[0]
else:
    best_pair = coint_df.sort_values('p_value').iloc[0]

asset1 = best_pair['Asset1']
asset2 = best_pair['Asset2']
hedge_ratio = best_pair['hedge_ratio']

print(f"📊 Spread Analysis for Best Pair:")
print(f"   Asset 1: {asset1}")
print(f"   Asset 2: {asset2}")
print(f"   Hedge Ratio: {hedge_ratio:.4f}")
print(f"   p-value: {best_pair['p_value']:.4f}")

# Calculate spread
spread = calculate_spread(price_matrix[asset1], price_matrix[asset2], hedge_ratio)
zscore = calculate_zscore(spread, lookback=20)

In [ ]:
# Visualize spread and z-score
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Normalized Prices
p1_norm = price_matrix[asset1] / price_matrix[asset1].iloc[0] * 100
p2_norm = price_matrix[asset2] / price_matrix[asset2].iloc[0] * 100

axes[0].plot(p1_norm.index, p1_norm.values, label=asset1, linewidth=1.5, color='blue')
axes[0].plot(p2_norm.index, p2_norm.values, label=asset2, linewidth=1.5, color='orange')
axes[0].set_title(f'Normalized Prices: {asset1} vs {asset2}', fontweight='bold')
axes[0].set_ylabel('Price (Base=100)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Spread
axes[1].plot(spread.index, spread.values, linewidth=1, color='purple')
axes[1].axhline(y=spread.mean(), color='green', linestyle='--', label=f'Mean: {spread.mean():.2f}')
axes[1].fill_between(spread.index, spread.mean() - 2*spread.std(), spread.mean() + 2*spread.std(), 
                      alpha=0.2, color='green', label='±2 Std')
axes[1].set_title('Spread (Price1 - β × Price2)', fontweight='bold')
axes[1].set_ylabel('Spread')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Z-Score
axes[2].plot(zscore.index, zscore.values, linewidth=1, color='teal')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].axhline(y=2, color='red', linestyle='--', linewidth=1, label='Entry (+2)')
axes[2].axhline(y=-2, color='green', linestyle='--', linewidth=1, label='Entry (-2)')
axes[2].axhline(y=1, color='orange', linestyle=':', linewidth=1)
axes[2].axhline(y=-1, color='orange', linestyle=':', linewidth=1)
axes[2].fill_between(zscore.index, -2, 2, alpha=0.1, color='gray')
axes[2].set_title('Spread Z-Score (Trading Signal)', fontweight='bold')
axes[2].set_ylabel('Z-Score')
axes[2].set_xlabel('Date')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test for stationarity of spread
adf_result = adfuller(spread.dropna())
print(f"\n📊 Spread Stationarity Test (ADF):")
print(f"   ADF Statistic: {adf_result[0]:.4f}")
print(f"   p-value: {adf_result[1]:.4f}")
print(f"   {'✅ Spread is stationary (mean-reverting)' if adf_result[1] < 0.05 else '❌ Spread is non-stationary'}")

---

## 🎨 Step 7: Cluster Analysis

Let's identify groups of assets that move together using hierarchical clustering.

In [ ]:
# Hierarchical clustering based on correlation
print("📊 Hierarchical Clustering Analysis:")

# Convert correlation to distance
distance_matrix = 1 - corr_matrix

# Perform hierarchical clustering
condensed_dist = squareform(distance_matrix, checks=False)
linkage_matrix = linkage(condensed_dist, method='ward')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(16, 8))
dendrogram(linkage_matrix, labels=corr_matrix.columns, leaf_rotation=90, ax=ax)
ax.set_title('Hierarchical Clustering of Assets (Based on Correlation)', fontweight='bold', fontsize=14)
ax.set_xlabel('Asset')
ax.set_ylabel('Distance (1 - Correlation)')
plt.tight_layout()
plt.show()

In [ ]:
# Extract clusters
n_clusters = min(5, len(corr_matrix.columns) // 2)  # Aim for reasonable number of clusters
clusters = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

cluster_df = pd.DataFrame({
    'Asset': corr_matrix.columns,
    'Cluster': clusters
})

print(f"📊 Cluster Assignments ({n_clusters} clusters):")
for c in range(1, n_clusters + 1):
    assets_in_cluster = cluster_df[cluster_df['Cluster'] == c]['Asset'].tolist()
    print(f"\n   Cluster {c}: {assets_in_cluster}")

In [ ]:
# Visualize cluster correlations
# Sort assets by cluster
sorted_assets = cluster_df.sort_values('Cluster')['Asset'].tolist()
sorted_corr = corr_matrix.loc[sorted_assets, sorted_assets]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sorted_corr, cmap='RdBu_r', center=0, ax=ax,
            xticklabels=True, yticklabels=True, square=True,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix (Ordered by Cluster)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---

## 📐 Step 8: Pairs Trading Strategy Design

Based on our analysis, here's how we would implement a pairs trading strategy.

In [ ]:
def pairs_trading_backtest(price1, price2, hedge_ratio, lookback=20, 
                            entry_z=2.0, exit_z=0.5, stop_z=4.0):
    """
    Simple pairs trading backtest.
    
    Trading rules:
    - Long spread when z-score < -entry_z
    - Short spread when z-score > entry_z
    - Exit when z-score crosses exit_z towards zero
    - Stop-loss at stop_z
    
    Returns:
    - DataFrame with signals and returns
    """
    
    # Calculate spread and z-score
    spread = calculate_spread(price1, price2, hedge_ratio)
    zscore = calculate_zscore(spread, lookback)
    
    # Calculate returns
    r1 = price1.pct_change()
    r2 = price2.pct_change()
    
    # Initialize
    position = 0  # 1 = long spread, -1 = short spread, 0 = no position
    positions = []
    
    for i in range(len(zscore)):
        z = zscore.iloc[i]
        
        if pd.isna(z):
            positions.append(0)
            continue
        
        # Entry signals
        if position == 0:
            if z < -entry_z:  # Spread is too low, expect mean reversion up
                position = 1  # Long spread: long asset1, short asset2
            elif z > entry_z:  # Spread is too high, expect mean reversion down
                position = -1  # Short spread: short asset1, long asset2
        
        # Exit signals
        elif position == 1:  # Currently long spread
            if z > -exit_z or z > stop_z:  # Exit or stop
                position = 0
        
        elif position == -1:  # Currently short spread
            if z < exit_z or z < -stop_z:  # Exit or stop
                position = 0
        
        positions.append(position)
    
    # Create results DataFrame
    results = pd.DataFrame({
        'date': price1.index,
        'zscore': zscore.values,
        'position': positions
    })
    results.set_index('date', inplace=True)
    
    # Calculate strategy returns
    # Long spread = long asset1 + short asset2
    # Short spread = short asset1 + long asset2
    spread_return = r1 - hedge_ratio * r2
    results['spread_return'] = spread_return
    results['strategy_return'] = results['position'].shift(1) * results['spread_return']
    results['cumulative_return'] = (1 + results['strategy_return'].fillna(0)).cumprod()
    
    return results

print("✅ Pairs trading backtest function defined!")

In [ ]:
# Run pairs trading backtest for the best pair
print(f"📊 Pairs Trading Backtest: {asset1} vs {asset2}")
print("="*60)

pairs_results = pairs_trading_backtest(
    price_matrix[asset1],
    price_matrix[asset2],
    hedge_ratio,
    lookback=20,
    entry_z=2.0,
    exit_z=0.5
)

# Calculate metrics
strategy_returns = pairs_results['strategy_return'].dropna()

total_return = (pairs_results['cumulative_return'].iloc[-1] - 1) * 100
n_years = len(strategy_returns) / 252
ann_return = ((1 + total_return/100) ** (1/n_years) - 1) * 100 if n_years > 0 else 0
ann_vol = strategy_returns.std() * np.sqrt(252) * 100
sharpe = ann_return / ann_vol if ann_vol > 0 else 0

# Number of trades
position_changes = pairs_results['position'].diff().fillna(0) != 0
n_trades = position_changes.sum()

print(f"\n📈 Performance Metrics:")
print(f"   Total Return: {total_return:.2f}%")
print(f"   Annualized Return: {ann_return:.2f}%")
print(f"   Annualized Volatility: {ann_vol:.2f}%")
print(f"   Sharpe Ratio: {sharpe:.4f}")
print(f"   Number of Trades: {n_trades}")
print(f"   Win Rate: {(strategy_returns > 0).mean() * 100:.1f}%")

In [ ]:
# Visualize pairs trading results
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Z-Score and positions
axes[0].plot(pairs_results.index, pairs_results['zscore'], linewidth=1, color='teal', label='Z-Score')
axes[0].axhline(y=2, color='red', linestyle='--', alpha=0.5)
axes[0].axhline(y=-2, color='green', linestyle='--', alpha=0.5)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Mark positions
long_spread = pairs_results['position'] == 1
short_spread = pairs_results['position'] == -1

axes[0].fill_between(pairs_results.index, pairs_results['zscore'].min(), pairs_results['zscore'].max(),
                      where=long_spread, alpha=0.2, color='green', label='Long Spread')
axes[0].fill_between(pairs_results.index, pairs_results['zscore'].min(), pairs_results['zscore'].max(),
                      where=short_spread, alpha=0.2, color='red', label='Short Spread')

axes[0].set_title('Z-Score and Trading Positions', fontweight='bold')
axes[0].set_ylabel('Z-Score')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Daily Strategy Returns
axes[1].bar(pairs_results.index, pairs_results['strategy_return'] * 100, 
            color=np.where(pairs_results['strategy_return'] >= 0, 'green', 'red'), alpha=0.6, width=1)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_title('Daily Strategy Returns (%)', fontweight='bold')
axes[1].set_ylabel('Return (%)')
axes[1].grid(True, alpha=0.3)

# 3. Cumulative Returns
axes[2].plot(pairs_results.index, (pairs_results['cumulative_return'] - 1) * 100, 
             linewidth=2, color='purple', label='Pairs Strategy')
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[2].set_title('Cumulative Returns (%)', fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Cumulative Return (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Pairs Trading: {asset1} vs {asset2}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 📐 Step 9: Integration Proposal

How to incorporate the pairs trading signal into the main portfolio strategy.

### 💡 Integration Proposal: Relative Value Overlay

**Mathematical Framework:**

Let our main alpha signal be $\alpha_i$ for stock $i$, and our pairs signal be $\beta_{ij}$ for pair $(i,j)$.

**Combined Signal:**
$$w_i = \lambda \cdot \alpha_i + (1-\lambda) \cdot \sum_j \beta_{ij}$$

Where:
- $w_i$ = final weight for stock $i$
- $\alpha_i$ = momentum/ML-based alpha from main strategy
- $\beta_{ij}$ = pairs trading signal for stock $i$ with partner $j$
- $\lambda$ = blending parameter (e.g., 0.7 for 70% main, 30% pairs)

**Implementation Steps:**

1. **Identify Pairs:** Use cointegration test to select stable pairs
2. **Calculate Z-Scores:** For each pair, compute the spread z-score daily
3. **Generate Signals:** 
   - Z-score > 2: Short the expensive asset, long the cheap one
   - Z-score < -2: Long the expensive asset, short the cheap one
4. **Blend with Main Strategy:**
   - If main strategy says long AND pairs says long → Strong conviction
   - If signals disagree → Reduce position size or stay neutral
5. **Risk Management:**
   - Maximum allocation to pairs overlay: 30% of portfolio
   - Stop-loss at z-score > 4
   - Maximum holding period: 20 days

**Expected Benefits:**
- Reduces portfolio beta through market-neutral pairs trades
- Captures mean-reversion alpha orthogonal to momentum
- Provides diversification benefit when main strategy underperforms

In [ ]:
# Code demonstration for integration

def create_pairs_overlay_signal(prices_df, coint_pairs_df, ticker_col, date_col, close_col,
                                 max_pairs=5, entry_z=2.0):
    """
    Create pairs trading overlay signal for all stocks.
    
    Returns:
    - DataFrame with pairs overlay signal for each stock-date
    """
    
    price_pivot = prices_df.pivot(index=date_col, columns=ticker_col, values=close_col)
    
    # Select top cointegrated pairs
    top_pairs = coint_pairs_df.head(max_pairs)
    
    # Initialize signal matrix
    overlay_signals = pd.DataFrame(0.0, index=price_pivot.index, columns=price_pivot.columns)
    
    for _, row in top_pairs.iterrows():
        a1, a2 = row['Asset1'], row['Asset2']
        hr = row['hedge_ratio']
        
        if a1 not in price_pivot.columns or a2 not in price_pivot.columns:
            continue
        
        # Calculate z-score
        spread = price_pivot[a1] - hr * price_pivot[a2]
        zscore = (spread - spread.rolling(20).mean()) / spread.rolling(20).std()
        
        # Generate signals
        # Z-score > entry_z: short a1, long a2
        # Z-score < -entry_z: long a1, short a2
        
        long_signal = zscore < -entry_z
        short_signal = zscore > entry_z
        
        overlay_signals.loc[long_signal, a1] += 1
        overlay_signals.loc[long_signal, a2] -= hr
        
        overlay_signals.loc[short_signal, a1] -= 1
        overlay_signals.loc[short_signal, a2] += hr
    
    return overlay_signals

print("✅ Pairs overlay function defined!")

# Example usage (conceptual)
print("\n📌 Example Integration:")
print("""
# In your main trading loop:

# 1. Get main strategy signals
main_signals = generate_main_strategy_signals(features)

# 2. Get pairs overlay signals
pairs_signals = create_pairs_overlay_signal(prices, cointegrated_pairs, ...)

# 3. Blend signals
lambda_blend = 0.7  # 70% main, 30% pairs
combined_signals = lambda_blend * main_signals + (1 - lambda_blend) * pairs_signals

# 4. Normalize to target leverage
combined_signals = combined_signals / combined_signals.abs().sum(axis=1).max()

# 5. Execute trades
positions = combined_signals
""")

---

## 💾 Step 10: Save Analysis Results

In [ ]:
# Save all results

# Correlation matrix
corr_matrix.to_csv('correlation_matrix.csv')

# Cointegration results
coint_df.to_csv('cointegration_results.csv', index=False)

# Lead-lag results
lead_lag_df.to_csv('lead_lag_results.csv', index=False)

# Cluster assignments
cluster_df.to_csv('cluster_assignments.csv', index=False)

# Pairs trading backtest results
pairs_results.to_csv('pairs_trading_backtest.csv')

print("✅ All results saved!")
print("   - correlation_matrix.csv")
print("   - cointegration_results.csv")
print("   - lead_lag_results.csv")
print("   - cluster_assignments.csv")
print("   - pairs_trading_backtest.csv")

---

## 📝 Summary & Key Findings

### What We Accomplished:

1. **Correlation Analysis:**
   - Computed pairwise correlations for all assets
   - Identified most correlated and anti-correlated pairs
   - Visualized correlation structure

2. **Cointegration Testing:**
   - Tested all pairs for cointegration using Engle-Granger method
   - Calculated optimal hedge ratios for cointegrated pairs
   - Identified pairs suitable for mean-reversion trading

3. **Lead-Lag Analysis:**
   - Discovered which assets lead or lag others
   - Identified potential predictive relationships

4. **Cluster Analysis:**
   - Grouped assets by correlation similarity
   - Identified potential sector/factor groupings

5. **Pairs Trading Strategy:**
   - Designed a mean-reversion strategy for cointegrated pairs
   - Backtested the strategy with entry/exit rules
   - Calculated performance metrics

6. **Integration Proposal:**
   - Outlined mathematical framework for combining with main strategy
   - Provided implementation code structure

### Key Insights:

- High correlation doesn't guarantee cointegration
- Mean-reverting spreads provide a different alpha source than momentum
- Lead-lag relationships can enhance predictive power
- Pairs trading works best when spread is truly stationary

### Recommendations:

1. **Use cointegration over correlation** for pairs trading
2. **Regularly retest pairs** - relationships can break down
3. **Limit exposure** to any single pair (max 5-10% of pairs budget)
4. **Blend with main strategy** at 20-30% allocation

---

# 🎉 Project Complete!

This concludes the PRECOG Quant Task. All four parts of the pipeline are now implemented:

1. ✅ Part 1: Feature Engineering & Data Cleaning
2. ✅ Part 2: Model Training & Strategy Formulation
3. ✅ Part 3: Backtesting & Performance Analysis
4. ✅ Part 4: Statistical Arbitrage Overlay

---